# About
This notebook is a step by step cell run to build out the full C3S dataset from raw json sets into a .db file.

What you need are:
- src_data folder: where all the json files live
- C3SDB_schema : for sqlite schema of features
- mqn_schema : for sqlite schema of MQNs
- pred_CCS_scema : for sqlite schema of predicted CCS
- build_utils : folder for extra functions to build out the db file

In [1]:
import sqlite3
import os
import pandas as pd 
import requests

from build_utils.db_init import create_db
from build_utils.src_data import add_dataset
from build_utils.smiles import (
    load_smiles_search_cache,
    save_smiles_search_cache,
    add_smiles_to_db,
)
from build_utils.mqns import add_mqns_to_db
from build_utils.classification import label_class_byname
from build_utils.clean_src import clean_database, remove_invalid_smiles, create_clean_db


# Initialize DB
Define a new db file with schema

In [ ]:
#Initialize New DB File
db_name = "C3S.db"
create_db(db_name)

#Connect to DB file
con = sqlite3.connect(db_name)
cur = con.cursor()


# Populating DB File From JSON Datasets
1. adding all the raw data
2. adding smiles structures from pubchem api
3. adding mqns
4. adding chemical classification

### adding raw json entries

In [ ]:
# source datasets to include in the raw json files 
_SRC_TAGS = [
    "zhou1016",
    "zhou0817",
    "zhen0917",
    "pagl0314",
    "righ0218",
    "nich1118",
    "may_0114",
    "moll0218",
    "hine1217",
    "hine0217",
    "hine0817",
    "groe0815",
    "bijl0517",
    "stow0817",
    "hine0119",
    "leap0219",
    "blaz0818",
    # "vasi0120", exclude vasi
    "tsug0220",
    "lian0118",
    "teja0918",
    "pola0620",
    "dodd0220",
    "celm1120",
    "belo0321",
    "ross0422",
    "baker0524", #new
    "mull_1223", #new
    "palm_0424", #new
    "extra_ross0422", #new
]

In [4]:
n_entries = 0
for src_tag in _SRC_TAGS:
    n_added = add_dataset(cur, src_tag)
    n_entries += n_added
    print(f"\tsrc_tag: {src_tag} n_added: {n_added}")

print(f"\ttotal entries: {n_entries}")

	src_tag: zhou1016 n_added: 847
	total entries: 847


### adding smiles structures

In [5]:
smiles_cache_file = "smiles_search_cache.json"

# if a local copy of the SMILES search cache does not exist, grab the
# built-in copy from the package
if not os.path.isfile(smiles_cache_file):
    smiles_search_cache = load_smiles_search_cache(cache_file_name=None)
else:
    # load the local copy if it exists
    smiles_search_cache = load_smiles_search_cache(
        cache_file_name=smiles_cache_file 
    )

# Initialize session
sess = requests.Session()

# Add SMILES structures to DB
n_smiles, n_requests = add_smiles_to_db(cur, sess, smiles_search_cache)
print(f"\tSMILES structures added: {n_smiles}")
print(f"\tweb requests sent: {n_requests}")
print("... done")

	(   169) CCSBASE_EDC0718C9B Reduced nicotinamide adenine dinucleotide (NADH)                                                    An error occurred: 404 Client Error: PUGREST.NotFound for url: https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/Reduced%20nicotinamide%20adenine%20dinucleotide%20(NADH)/cids/TXT
An error occurred: 404 Client Error: PUGREST.NotFound for url: https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/Reduced%20nicotinamide%20adenine%20dinucleotide%20(NADH)/cids/TXT
	(   460) CCSBASE_13E7051CFC D-(+)-Melibiose                                                                                     An error occurred: 404 Client Error: PUGREST.NotFound for url: https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/D-(+)-Melibiose/cids/TXT
An error occurred: 404 Client Error: PUGREST.NotFound for url: https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/D-(+)-Melibiose/cids/TXT
	(   655) CCSBASE_2B2853E510 P1,P4-Diadenosine 5'-tetraphosphate                    

### adding mqn

In [6]:
print("adding MQNs to database entries ...")
n_mqns = add_mqns_to_db(cur)
print(f"\tentries with MQNs: {n_mqns}")
print("... done")

adding MQNs to database entries ...
	entries with MQNs: 840
... done


### adding chemical labels 

In [7]:
print("adding rough chemical classification labels to database entries ...")
label_class_byname(cur)

adding rough chemical classification labels to database entries ...


In [8]:
con.commit()
con.close()

# Making Cleaned DB version
1. cleaning invalid smiles structures
2. clean code by DT and TW values
3. cleaning by relative standard deviation


In [ ]:
#open c3s db file 
db_name = "C3S.db"
clean_db_name = "C3S_clean_smile.db"

create_clean_db(clean_db_name)

In [10]:
# 1. remove invalid smiles
remove_invalid_smiles(db_name, clean_db_name) #updates the clean DB with only valid smiles

In [ ]:
# 2. clean code by DT and TW values and rsd
clean_database("C3S_clean_smile.db", "C3S_clean_smile_rsd.db")

Creating new clean database: C3S_clean_3.db
🔴 Entries processed: 0/830 🔴 
🔴 Entries processed: 1/830 🔴 
✅ Processing group with key: ('(s)-2-hydroxyglutarate', '[M-H]-', np.float64(147.0))                    g_id                    name  adduct      mass  z  \
563  CCSBASE_5641F26339  (s)-2-hydroxyglutarate  [M-H]-  147.0294  1   
564  CCSBASE_6EAFC770D0  (s)-2-hydroxyglutarate  [M-H]-  147.0294  1   

           mz  ccs                        smi chem_class_label   src_tag  ...  \
563  147.0294  118  C(CC(=O)O)[C@@H](C(=O)O)O   small molecule  zhou1016  ...   
564  147.0294  119  C(CC(=O)O)[C@@H](C(=O)O)O   small molecule  zhou1016  ...   

    r4 r5  r6  r7  r8  r9  rg10  afr  bfr  rounded_mz  
563  0  0   0   0   0   0     0    0    0       147.0  
564  0  0   0   0   0   0     0    0    0       147.0  

[2 rows x 55 columns] and size: 2
converting into dictonary
processing two entries
two entries: RSD less than 1
🟠 Processed 1 entries 🟠
🟠 Inserted processed group with key: ('(s)-2-